# Logistic Regression Pipeline

This notebook adapts our spatial decoding framework to use a Logistic Regression classifier.

Operating on the same normalized voxel-wise labels, we train a standard pipeline (feature scaling + logistic regression) across 5-fold cross-validation. For each fold, we optimize the inverse regularization strength (`C`) via Grid Search and evaluate the best-fitting model on unseen test coordinates.

In [2]:
from pathlib import Path
import numpy as np
from time import time
import scipy.io as sio
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from sklearn.utils import shuffle
import joblib
from scipy.sparse import eye as sparse_eye
from tqdm import trange
from sklearn.svm import LinearSVC
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from utils import calculate_laplacian_matrix, calculate_Sn_new
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


DATA_PATH = Path('../data')
RESULT_PATH = Path('../results')
BRAIN_HEMISPHERE_LIST = ['left', 'right']
LAMBDA_REG = 1e-1 # # balance smooth level
RADIUS = 2.5 # smooth radius
MARGIN = 0.7 # gap between voxels grids

# Model Training: Fit Logistic Regression

We train the Logistic Regression model across 5-fold cross-validation. For each fold, the script spatially normalizes the training peak coordinates, utilizes a Grid Search to find the optimal inverse regularization strength (`C`), and tests the best-fitting estimator against the unseen test coordinates.

In [ ]:
raw_data_path = DATA_PATH / 'raw_data.dict.pkl'
raw_data_dict = joblib.load(raw_data_path)
train_test_idx_path = DATA_PATH / 'train_test_index.dict.pkl'
train_test_idx_dict = joblib.load(train_test_idx_path)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(        solver='lbfgs',        max_iter=10000,    )),
])

param_grid = {'logreg__C': [0.01, 0.1, 1.0, 10.0, 100.0]}
grid_search = GridSearchCV(
    estimator=model, param_grid=param_grid,
    cv=5, scoring='accuracy', n_jobs=5
)


In [ ]:
for brain_hemisphere in BRAIN_HEMISPHERE_LIST:
    acc_list = []
    peak_wide = raw_data_dict[f'{brain_hemisphere}_peak']
    for fold_i in range(5):
        (train_idx, test_idx) = train_test_idx_dict[(brain_hemisphere, fold_i)]
        print(f'Now processing: {brain_hemisphere} - fold{fold_i}')

        # 1. Get X train
        X_train = raw_data_dict[f'{brain_hemisphere}_MNI']

        # 2. Get y train
        ## load mni grid data
        mni_loc = raw_data_dict[f'{brain_hemisphere}_MNI']
        print(f'N points in MNI: {len(mni_loc)}')
        peak_loc = peak_wide[train_idx, :]
        print(f'N subjects: {len(peak_loc)}')
        print('Now calculating the Laplacian Matrix, might take a while...')

        L = raw_data_dict[f'{brain_hemisphere}_laplacian_matrix']
        ## Spatial Normalization for each voxel
        Sn = calculate_Sn_new(peak_loc, mni_loc, r)
        Sn = csc_matrix(Sn)
        I = sparse_eye(L.shape[0], format='csc')
        norm_matrix = lambda_reg * spsolve(L + lambda_reg * I, Sn)
        ## predict task labels
        ## +1: Python index starts from 0, while MATLAB start from 1
        y_train = np.argmax(norm_matrix.toarray(), axis=1) + 1

        # 3. Get X test and y test
        peak_wide_test = peak_wide[test_idx, :]
        X_test = np.vstack([peak_wide_test[:, 0:3], peak_wide_test[:, 3:6], peak_wide_test[:, 6:9]])
        y_test = np.repeat(np.arange(1, 4), peak_wide_test.shape[0])

        print(f'X train: {X_train.shape}, y train: {y_train.shape}')
        print(f'X test: {X_test.shape} y test: {y_test.shape}')

        # 4. Fit Model
        # hyperparameter tuning
        grid_search.fit(X_train, y_train)
        print("Best parameters found: ", grid_search.best_params_)
        y_predict = grid_search.best_estimator_.predict(X_test)
        acc = (y_predict == y_test).mean()
        print(' Accuracy:', acc)
        acc_list.append(acc)
        fitted_grid_search_file_path = RESULT_PATH / f'{brain_hemisphere}_fold{fold_i}_Logistic_GridSearch.pkl'
        joblib.dump(grid_search, fitted_grid_search_file_path)

    acc_list_file = RESULT_PATH / f'{brain_hemisphere}_Logistic_acc.pkl'
    joblib.dump(acc_list, acc_list_file)

# Permutation Test

To evaluate the statistical significance of our logistic model, we perform a standard permutation test. Using the optimal hyperparameters identified during cross-validation, we retrain the classifier on datasets with randomly shuffled task labels. This generates a null distribution of accuracy, establishing the baseline chance level for our spatial decoding.

In [9]:
hyper_parameter_list = dict()
for brain_hemisphere in BRAIN_HEMISPHERE_LIST:
    best_C_list = []
    for fold_i in range(5):
        gs_path = RESULT_PATH / f'{brain_hemisphere}_fold{fold_i}_Logistic_GridSearch.pkl'
        gs = joblib.load(gs_path)
        best_C_list.append(gs.best_params_['logreg__C'])
    hyper_parameter_list[brain_hemisphere] = best_C_list
hyper_parameter_list

{'left': [10.0, 100.0, 10.0, 10.0, 1.0], 'right': [10.0, 100.0, 0.1, 0.1, 0.1]}

In [12]:
for brain_hemisphere in ['left', 'right']:
    acc_list_file = RESULT_PATH / f'{brain_hemisphere}_Logistic_permutation_acc.pkl'
    if acc_list_file.exists():
        continue
    permutation_data_path = DATA_PATH / f'{brain_hemisphere}_permutation.list.pkl'
    permutation_data_dict_list = joblib.load(permutation_data_path)
    permutation_acc_list = []
    for permutation_data_dict in tqdm(permutation_data_dict_list):
        fold_i = permutation_data_dict['fold_i']

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('logreg', LogisticRegression( solver='lbfgs', max_iter=10000, )),
        ])
        X_train = raw_data_dict[f'{brain_hemisphere}_MNI']
        model.fit(X_train, permutation_data_dict['y_train'])
        y_predict = model.predict(permutation_data_dict['X_test'])
        acc = (y_predict == permutation_data_dict['y_test']).mean()
        permutation_acc_list.append(acc)

    joblib.dump(permutation_acc_list, acc_list_file)